In [6]:
print("Hola, mundo!")

Hola, mundo!


In [1]:
# -*- coding: utf-8 -*-
# Importar las bibliotecas necesarias
import requests
from bs4 import BeautifulSoup
import psycopg2
import pandas as pd
from sqlalchemy import create_engine  # Importa create_engine

# 1. Configuración de la base de datos PostgreSQL
# Reemplaza con tus propios detalles de conexión
db_host = "localhost"  # o la dirección IP de tu servidor PostgreSQL
db_name = "cybersecurity_logs"
db_user = "postgres"
db_password = "postgres"
db_port = 5432  # Puerto predeterminado de PostgreSQL
 
# 2. Función para conectar a la base de datos
def conectar_a_db():
    try:
        conn = psycopg2.connect(
            host=db_host,
            database=db_name,
            user=db_user,
            password=db_password,
            port=db_port
        )
        print("Conexión a la base de datos exitosa")
        return conn
    except psycopg2.Error as e:
        print(f"Error al conectar a la base de datos: {e}")
        return None

# 3. Función para crear la tabla (si no existe)
def crear_tabla(conn, tabla_nombre, columnas):
    try:
        cursor = conn.cursor()
        columnas_sql = ", ".join([f"{col} {tipo}" for col, tipo in columnas.items()])
        sql = f"""
            CREATE TABLE IF NOT EXISTS {tabla_nombre} (
                {columnas_sql}
            );
        """
        cursor.execute(sql)
        conn.commit()
        print(f"Tabla '{tabla_nombre}' creada o ya existe.")
        cursor.close()
    except psycopg2.Error as e:
        print(f"Error al crear la tabla: {e}")
        if conn:
            conn.rollback() # Retrocede la transacción en caso de error

# 4. Función para hacer scraping y extraer datos (modificada para retornar una lista de diccionarios)
def hacer_scraping(url, selector_padre, selector_hijo, atributos_a_extraer=None): # atributos_a_extraer es un diccionario {nombre_columna: "atributo_html"}
    """
    Realiza scraping en una página web y extrae datos.  Retorna una lista de diccionarios.

    Args:
        url (str): La URL de la página web a scrapear.
        selector_padre (str): Selector CSS del elemento padre.
        selector_hijo (str): Selector CSS del elemento hijo (el dato a extraer).
        atributos_a_extraer (dict, optional): Un diccionario donde la clave es el nombre de la columna y el valor es el atributo HTML a extraer (ej. {"enlace": "href"}). Defaults to None.

    Returns:
        list: Una lista de diccionarios, donde cada diccionario representa una fila de datos.
    """
    try:
        response = requests.get(url)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')
        elementos_padre = soup.select(selector_padre)
        resultados = []
        for elemento_padre in elementos_padre:
            elementos_hijo = elemento_padre.select(selector_hijo)
            for elemento_hijo in elementos_hijo:
                fila = {}
                fila["dato_extraido"] = elemento_hijo.text.strip()
                if atributos_a_extraer:
                    for nombre_columna, atributo in atributos_a_extraer.items():
                        valor = elemento_hijo.get(atributo)  # Extrae el atributo del elemento
                        fila[nombre_columna] = valor.strip() if valor else None # Si no existe, que retorne none.
                resultados.append(fila)
        return resultados
    except requests.exceptions.RequestException as e:
        print(f"Error al realizar la solicitud HTTP: {e}")
        return []
    except Exception as e:
        print(f"Error durante el scraping: {e}")
        return []



#sql alchemy
def insertar_datos_con_sqlalchemy(conn, tabla_nombre, df):
    """Inserta datos usando SQLAlchemy."""
    try:
        # Crea un engine de SQLAlchemy para PostgreSQL
        engine = create_engine(f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")

        # Inserta el DataFrame usando SQLAlchemy
        df.to_sql(tabla_nombre, engine, if_exists="append", index=False)
        print(f"{len(df)} filas insertadas en la tabla '{tabla_nombre}' usando SQLAlchemy.")

        # Cierra la conexión del engine (importante)
        engine.dispose()

    except Exception as e:
        print(f"Error al insertar datos con SQLAlchemy: {e}")
        if conn:
            conn.rollback()

# 5. Función para insertar datos en la base de datos (modificada para usar pandas)
def insertar_datos_con_pandas(conn, tabla_nombre, df):
    """
    Inserta datos en la base de datos utilizando pandas.

    Args:
        conn: Conexión a la base de datos.
        tabla_nombre (str): Nombre de la tabla.
        df (pd.DataFrame): DataFrame de pandas con los datos a insertar.
    """
    try:
        df.to_sql(tabla_nombre, conn, if_exists="append", index=False) # "append" para agregar datos, "replace" para reemplazar la tabla
        print(f"{len(df)} filas insertadas en la tabla '{tabla_nombre}' usando pandas.")
    except Exception as e:
        print(f"Error al insertar datos con pandas: {e}")
        if conn:
            conn.rollback()

# 6. Función principal (modificada)
def main():
    # 1. Conectar a la base de datos
    conn = conectar_a_db()
    if not conn:
        return

    # 2. Configuración del scraping

# Configuración del scraping
    url = "https://quotes.toscrape.com/"
    selector_padre = "div.quote"
    selector_hijo = "a"  # Selecciona la etiqueta <a> para el enlace
    atributos_a_extraer = {"enlace": "href"}
    tabla_nombre = "scraping_productos"

    


    # 3. Definir la estructura de la tabla (columnas y tipos de datos)
    columnas_tabla = {
        "id": "SERIAL PRIMARY KEY",
        "dato_extraido": "TEXT",
        "enlace": "TEXT",
        "precio": "NUMERIC",
        "fecha_hora": "TIMESTAMP DEFAULT CURRENT_TIMESTAMP"
    }


    # 4. Crear la tabla (si no existe)
    crear_tabla(conn, tabla_nombre, columnas_tabla)





    # 5. Realizar scraping
    resultados = hacer_scraping(url, selector_padre, selector_hijo, atributos_a_extraer) # Incluye atributos_a_extraer

    if resultados:
        # 6. Convertir los datos a un DataFrame de pandas
        df = pd.DataFrame(resultados)

        # Ajustar los nombres de las columnas para que coincidan con la base de datos (opcional)
        # df.rename(columns={"dato_extraido": "nombre"}, inplace=True)
        # Convierte el precio a tipo numérico (si es necesario)
        if "datoextraido" in df.columns:
            df["datoextraido"] = pd.to_numeric(df["precio"], errors='coerce')  # 'coerce' convierte errores a NaN

        # Limpiar el DataFrame (opcional)
        df.fillna("", inplace=True)  # Rellena NaN con cadenas vacías

        # Mostrar el DataFrame en Jupyter Notebook
        display(df)

        # 7. Insertar datos en la base de datos usando pandas
        #insertar_datos_con_pandas(conn, tabla_nombre, df)
        insertar_datos_con_sqlalchemy(conn, tabla_nombre, df)
    else:
        print("No se encontraron datos para insertar.")

    # 8. Cerrar la conexión a la base de datos
    if conn:
        conn.close()
        print("Conexión a la base de datos cerrada.")

if __name__ == "__main__":
    main()


Conexión a la base de datos exitosa
Tabla 'scraping_productos' creada o ya existe.


,dato_extraido,enlace
0,(about),/author/Albert-Einstein
1,change,/tag/change/page/1/
2,deep-thoughts,/tag/deep-thoughts/page/1/
3,thinking,/tag/thinking/page/1/
4,world,/tag/world/page/1/
5,(about),/author/J-K-Rowling
6,abilities,/tag/abilities/page/1/
7,choices,/tag/choices/page/1/
8,(about),/author/Albert-Einstein
9,inspirational,/tag/inspirational/page/1/


40 filas insertadas en la tabla 'scraping_productos' usando SQLAlchemy.
Conexión a la base de datos cerrada.
